# Araç Fiyat Tahmini

Bu projede ikinci el araç fiyatını tahmin eden bir model kuracağım. Hedefim fiyatı mümkün olduğunca doğru tutturmak.


In [ ]:
import pandas as pd
pd.set_option('display.max_columns',100)

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns


### Data Dosyasını Okuma


In [ ]:
df=pd.read_csv('data/CarPrice.csv')
df.head()


### EDA - Exploraty Data Analysis


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


In [ ]:
df.describe()


### Veri Görselleştirme


In [ ]:
sns.heatmap(df.corr(numeric_only=True),annot=False)
plt.show()


In [ ]:
sns.scatterplot(x='enginesize',y='price',data=df,hue='fueltype')
plt.show()


### Boş Verileri Doldurma


In [ ]:
# bu sette boş yok ama yine de numerikleri median ile doldurdum
for c in df.select_dtypes('number').columns:
    df[c]=df[c].fillna(df[c].median())


### Feature Engineering


In [ ]:
df['marka']=df['CarName'].astype(str).str.split().str[0].str.lower()
df['marka']=df['marka'].replace({'maxda':'mazda','toyouta':'toyota','porcshce':'porsche','vokswagen':'volkswagen','vw':'volkswagen'})

x=df[['enginesize','horsepower','citympg','curbweight','fueltype','carbody']]
y=df['price']
x=pd.get_dummies(x,drop_first=True)
x.shape


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=42)


### Model - en az 3 tane denedim


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error

modeller={
    'Linear':LinearRegression(),
    'DecisionTree':DecisionTreeRegressor(random_state=42),
    'RandomForest':RandomForestRegressor(n_estimators=200,random_state=42)
}

sonuc=[]
for ad,m in modeller.items():
    m.fit(x_train,y_train)
    p=m.predict(x_test)
    sonuc.append((ad,r2_score(y_test,p),mean_absolute_error(y_test,p)))
    print(ad,'R2',round(r2_score(y_test,p),3),'MAE',round(mean_absolute_error(y_test,p),1))


### En iyi model - Feature Importance ve Residual


In [ ]:
rf=modeller['RandomForest']
pred=rf.predict(x_test)

imp=pd.DataFrame({'ozellik':x.columns,'onem':rf.feature_importances_}).sort_values('onem',ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x='onem',y='ozellik',data=imp.head(10))
plt.title('Feature Importance')
plt.show()

artik=y_test-pred
plt.scatter(pred,artik)
plt.axhline(0,color='red')
plt.xlabel('Tahmin')
plt.ylabel('Residual')
plt.show()


In [ ]:
import joblib
joblib.dump(rf,'../../models/regression_car_price.joblib')


### Sonuç

Gerçek CarPrice verisiyle 3 model denedim. Random Forest daha iyi geldi. Engine size ve beygir gücü fiyatı en çok etkiliyor. Residual grafikte uç değerler var ama genel olarak hedefi tutturdum, fiyat tahmini işe yarıyor.
